In [0]:
import pyspark.sql.functions as F

In [0]:
raw_county = spark.read.table("digital_redlining.bronze_census.raw_demographic_county")
raw_state = spark.read.table("digital_redlining.bronze_census.raw_demographic_state")
raw_msa = spark.read.table("digital_redlining.bronze_census.raw_demographic_msa")

In [0]:
# Add a geo_level column to each dataframe
raw_county = raw_county.withColumn("geo_level", F.lit("county"))
raw_state = raw_state.withColumn("geo_level", F.lit("state"))
raw_msa = raw_msa.withColumn("geo_level", F.lit("msa"))

In [0]:
# Drop unnecessary columns
raw_county = raw_county.select(["GEO_ID", "NAME", "geo_level", "P1_001N"])
raw_state = raw_state.select(["GEO_ID", "NAME", "geo_level", "P1_001N"])
raw_msa = raw_msa.select(["GEO_ID", "NAME", "geo_level", "P1_001N"])

In [0]:
# Union the raw tables
raw_combined = raw_county.union(raw_state).union(raw_msa)

In [0]:
# Clean the data
df = (
    raw_combined
    .withColumnRenamed("GEO_ID", "geo_id")
    .withColumnRenamed("NAME", "location_name")
    .withColumnRenamed("P1_001N", "total_population")
)
# Drop the prefix from geoids
df = df.withColumn("geo_id", F.expr("substring(geo_id, 10, length(geo_id))"))
# Add an "id" column
df = df.withColumn("id", F.monotonically_increasing_id())
df = df.select(['id', 'geo_id', 'location_name', 'geo_level'])

In [0]:
display(df)

In [0]:
df.count()

In [0]:
df = df.drop_duplicates(["geo_id"])


In [0]:
df.count()